# SALBA ML System - Notebook 2: Feature Engineering

## Objective
Create meaningful features from raw data for ML models.

## Feature Categories
1. Text-based features
2. Location-based features
3. Temporal features
4. Quality indicators

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Libraries loaded")

In [ ]:
# Load data
df = pd.read_csv('data/training_data.csv')
print(f"Loaded {len(df)} records")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
# 1. TEXT-BASED FEATURES
print("\n" + "="*60)
print("TEXT-BASED FEATURES")
print("="*60)

df['note'] = df['note'].fillna('')
df['text_length'] = df['note'].str.len()
df['word_count'] = df['note'].str.split().str.len()

# Urgency keywords
urgency_keywords = ['urgent', 'critical', 'emergency', 'immediate', 'severe']
df['has_urgency'] = df['note'].str.lower().str.contains('|'.join(urgency_keywords), na=False).astype(int)
df['urgency_count'] = df['note'].str.lower().str.findall('|'.join(urgency_keywords)).str.len()

# Prank keywords
prank_keywords = ['prank', 'test', 'fake', 'hoax', 'false']
df['has_prank_keywords'] = df['note'].str.lower().str.contains('|'.join(prank_keywords), na=False).astype(int)

print(f"\nText features created:")
print(f"  - text_length (range: {df['text_length'].min()}-{df['text_length'].max()})")
print(f"  - word_count (range: {df['word_count'].min()}-{df['word_count'].max()})")
print(f"  - has_urgency ({df['has_urgency'].sum()} reports)")
print(f"  - has_prank_keywords ({df['has_prank_keywords'].sum()} reports)")

In [ ]:
# 2. LOCATION-BASED FEATURES
print("\n" + "="*60)
print("LOCATION-BASED FEATURES")
print("="*60)

# Normalize coordinates
lat_mean, lat_std = df['latitude'].mean(), df['latitude'].std()
lng_mean, lng_std = df['longitude'].mean(), df['longitude'].std()

df['lat_normalized'] = (df['latitude'] - lat_mean) / lat_std if lat_std > 0 else 0
df['lng_normalized'] = (df['longitude'] - lng_mean) / lng_std if lng_std > 0 else 0

# Distance from city center (Malaybalay roughly at 8.156, 125.124)
city_center = (8.1575, 125.1276)
df['distance_from_center'] = np.sqrt(
    (df['latitude'] - city_center[0])**2 + 
    (df['longitude'] - city_center[1])**2
) * 111  # ~111km per degree

print(f"Location features created:")
print(f"  - lat_normalized (mean: {df['lat_normalized'].mean():.2f})")
print(f"  - lng_normalized (mean: {df['lng_normalized'].mean():.2f})")
print(f"  - distance_from_center (mean: {df['distance_from_center'].mean():.2f} km)")

In [ ]:
# 3. TEMPORAL FEATURES
print("\n" + "="*60)
print("TEMPORAL FEATURES")
print("="*60)

df['created_at'] = pd.to_datetime(df['created_at'])
df['hour'] = df['created_at'].dt.hour
df['month'] = df['created_at'].dt.month
df['day_of_week'] = df['created_at'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_night'] = ((df['hour'] >= 20) | (df['hour'] < 6)).astype(int)

print(f"Temporal features created:")
print(f"  - hour (range: 0-23)")
print(f"  - month (range: 1-12)")
print(f"  - day_of_week (0=Monday, 6=Sunday)")
print(f"  - is_weekend ({df['is_weekend'].sum()} reports)")
print(f"  - is_night ({df['is_night'].sum()} reports)")

In [ ]:
# 4. QUALITY INDICATORS
print("\n" + "="*60)
print("QUALITY INDICATORS")
print("="*60)

# Report quality score (0-1)
df['quality_score'] = (
    (df['text_length'] > 20).astype(float) * 0.3 +  # Has meaningful text
    (df['has_urgency'] > 0).astype(float) * 0.3 +   # Shows urgency
    (df['has_prank_keywords'] == 0).astype(float) * 0.4  # No prank indicators
)

# Legitimacy indicator (inverse of prank keywords)
df['legitimacy_score'] = 1.0 - (df['has_prank_keywords'] * 0.5)

print(f"Quality indicators created:")
print(f"  - quality_score (range: {df['quality_score'].min():.2f}-{df['quality_score'].max():.2f})")
print(f"  - legitimacy_score (mean: {df['legitimacy_score'].mean():.2f})")

In [ ]:
# Visualize feature correlations
feature_cols = ['text_length', 'word_count', 'has_urgency', 'distance_from_center', 
                 'hour', 'month', 'quality_score']

plt.figure(figsize=(10, 8))
correlation_matrix = df[feature_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, fmt='.2f')
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for each disaster type
feature_importance = pd.DataFrame()

for disaster_type in df['disaster_type'].unique():
    subset = df[df['disaster_type'] == disaster_type]
    feature_importance[disaster_type] = {
        'avg_text_length': subset['text_length'].mean(),
        'pct_urgent': (subset['has_urgency'].sum() / len(subset)) * 100,
        'avg_quality': subset['quality_score'].mean(),
        'avg_distance_km': subset['distance_from_center'].mean()
    }

print("\nFeature characteristics by disaster type:")
print(feature_importance.T.round(2))

In [ ]:
# Save engineered features
df.to_csv('data/features_engineered.csv', index=False)
print("✅ Engineered features saved to data/features_engineered.csv")

# Display final feature set
engineered_features = [
    'text_length', 'word_count', 'has_urgency', 'urgency_count',
    'has_prank_keywords', 'lat_normalized', 'lng_normalized',
    'distance_from_center', 'hour', 'month', 'day_of_week',
    'is_weekend', 'is_night', 'quality_score', 'legitimacy_score'
]

print(f"\n📊 Total engineered features: {len(engineered_features)}")
print("\nFeature list:")
for i, feat in enumerate(engineered_features, 1):
    print(f"  {i:2d}. {feat}")

In [ ]:
print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE")
print("="*60)
print(f"✅ {len(engineered_features)} features created")
print(f"✅ Features saved to CSV")
print("\nNext: Run notebook 03 for model training")